# Robustness Analysis: Distance to Pareto Front & ε-Near-Pareto Fraction

For each drift detector and dataset, this notebook computes:

1. **Average Distance to the Pareto Front** — how far each sampled configuration is from the Pareto-optimal front (mean, median, 90th percentile).
2. **Fraction of ε-Near-Pareto Points** — the proportion of configurations within ε of the Pareto front.

Results are shown per dataset and aggregated across datasets with box-whisker plots.

In [20]:
import os
import glob
import re
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

In [21]:
# ── Configuration ────────────────────────────────────────────────────────────
CSV_DIR = 'results'

# ε tolerances for the near-Pareto fraction (in normalised-distance units)
EPSILONS = [0.01, 0.025, 0.05, 0.10]

# If True, measure each detector against its OWN Pareto front (robustness).
# If False, measure against the GLOBAL Pareto front across all detectors.
USE_OWN_PARETO_FRONT = True

# Outlier removal: remove points with accuracy or runtime beyond
# Q1 - IQR_FACTOR*IQR  or  Q3 + IQR_FACTOR*IQR  per detector per dataset.
REMOVE_OUTLIERS = True
IQR_FACTOR = 1.5

# Display name mapping (filename detector -> plot label)
DISPLAY_NAMES = {
    'EWDD': 'MoPEDDs',
}

# Detector colours and markers (consistent with plot_pareto_fronts.ipynb)
DETECTOR_COLORS = {
    'BNDM': '#1f77b4',
    'CSDDM': '#ff7f0e',
    'D3': '#2ca02c',
    'IBDD': '#d62728',
    'OCDD': '#9467bd',
    'SPLL': '#8c564b',
    'UDetect': '#e377c2',
    'MoPEDDs': '#17becf',
    'MoPEDDs_PreSelected': '#bcbd22',
}

## 1  Load result CSVs

In [31]:
csv_files = glob.glob(os.path.join(CSV_DIR, '*.csv'))

# data[dataset][detector] = DataFrame with at least 'accuracy' and 'runtime'
data = defaultdict(dict)

for path in sorted(csv_files):
    fname = os.path.basename(path)
    # Try PreSelected pattern first: e.g. EWDD_Electricity_PreSelected.csv
    match_pre = re.match(r'^(.+?)_(.+)_PreSelected\.csv$', fname)
    if match_pre:
        continue
        detector = 'MoPEDDs_PreSelected'
        dataset = match_pre.group(2)
    else:
        match = re.match(r'^(.+?)_(.+)\.csv$', fname)
        if not match:
            continue
        detector, dataset = match.group(1), match.group(2)
        detector = DISPLAY_NAMES.get(detector, detector)
    if dataset == "ForestCovertype":
        continue
    df = pd.read_csv(path)
    if 'accuracy' in df.columns and 'runtime' in df.columns:
        # Drop failed trials (accuracy=0, runtime=inf)
        df = df[(df['accuracy'] > 0) & (df['runtime'] < float('inf'))].copy()
        if len(df) > 0:
            if REMOVE_OUTLIERS:
                before = len(df)
                df = remove_iqr_outliers(df, factor=IQR_FACTOR)
                removed = before - len(df)
                if removed > 0:
                    print(f'  {detector}/{dataset}: removed {removed} outlier(s) of {before}')
            data[dataset][detector] = df

print(f'Found data for {len(data)} dataset(s):')
for ds in sorted(data):
    det_counts = {d: len(df) for d, df in sorted(data[ds].items())}
    print(f'  {ds}: {det_counts}')

## 2  Helper functions

In [23]:
def compute_pareto_front(acc, rt):
    """Compute the Pareto front (maximise accuracy, minimise runtime).

    Parameters
    ----------
    acc : array-like, shape (n,)
    rt  : array-like, shape (n,)

    Returns
    -------
    pareto_mask : boolean array of length n (True = Pareto-optimal)
    """
    acc = np.asarray(acc)
    rt = np.asarray(rt)
    n = len(acc)
    pareto_mask = np.ones(n, dtype=bool)
    for i in range(n):
        if not pareto_mask[i]:
            continue
        for j in range(n):
            if i == j or not pareto_mask[j]:
                continue
            # j dominates i if j is at least as good in both and strictly better in one
            if acc[j] >= acc[i] and rt[j] <= rt[i]:
                if acc[j] > acc[i] or rt[j] < rt[i]:
                    pareto_mask[i] = False
                    break
    return pareto_mask


def min_distance_to_pareto(points_norm, pareto_norm):
    """Euclidean distance from each point to the nearest Pareto-front point.

    Parameters
    ----------
    points_norm : (N, 2) array  – normalised (accuracy, runtime) for all points
    pareto_norm : (P, 2) array  – normalised (accuracy, runtime) for Pareto-optimal points

    Returns
    -------
    dists : (N,) array of minimum Euclidean distances
    """
    D = cdist(points_norm, pareto_norm, metric='euclidean')
    return D.min(axis=1)

def remove_iqr_outliers(df, factor=1.5):
    """Remove rows whose accuracy or runtime lies beyond the IQR fences.

    For each column in {accuracy, runtime}, rows outside
    [Q1 - factor*IQR, Q3 + factor*IQR] are dropped.
    """
    mask = pd.Series(True, index=df.index)
    for col in ['accuracy', 'runtime']:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - factor * iqr, q3 + factor * iqr
        mask &= (df[col] >= lo) & (df[col] <= hi)
    return df[mask].copy()

## 3  Compute distances to Pareto front per dataset

When  (default), each detector is measured against
its **own** Pareto front — this answers *"how easy is it to find a good config
for this detector?"*

When , all detectors are measured against the
**global** Pareto front (pooled across all detectors on that dataset).

In [24]:
# results_per_ds[dataset][detector] = dict with keys:
#   'distances'    : 1-D array of normalised distances to the Pareto front
#   'n_points'     : int
#   'mean_dist'    : float
#   'median_dist'  : float
#   'p90_dist'     : float
#   'eps_fractions' : dict {eps: fraction}
results_per_ds = {}

for dataset in sorted(data):
    detectors = data[dataset]

    # ── Pool all points ──────────────────────────────────────────────────
    all_acc = []
    all_rt = []
    all_labels = []  # detector name per row
    for det_name, df in detectors.items():
        all_acc.append(df['accuracy'].values)
        all_rt.append(df['runtime'].values)
        all_labels.extend([det_name] * len(df))
    all_acc = np.concatenate(all_acc)
    all_rt = np.concatenate(all_rt)
    all_labels = np.array(all_labels)

    # ── Normalise to [0, 1] ──────────────────────────────────────────────
    acc_min, acc_max = all_acc.min(), all_acc.max()
    rt_min, rt_max = all_rt.min(), all_rt.max()
    acc_range = acc_max - acc_min if acc_max > acc_min else 1.0
    rt_range = rt_max - rt_min if rt_max > rt_min else 1.0

    acc_norm = (all_acc - acc_min) / acc_range
    rt_norm = (all_rt - rt_min) / rt_range

    # For distance calculation we want "better = higher" for both axes,
    # so we flip runtime: low runtime → high value.
    rt_norm_flipped = 1.0 - rt_norm

    points_norm = np.column_stack([acc_norm, rt_norm_flipped])

    # ── Global Pareto front (used when USE_OWN_PARETO_FRONT is False) ────
    if not USE_OWN_PARETO_FRONT:
        pareto_mask = compute_pareto_front(all_acc, all_rt)
        global_pareto_points = points_norm[pareto_mask]
        print(f'\n{dataset}: {len(all_acc)} total points, '
              f'{pareto_mask.sum()} global Pareto-optimal')
    else:
        print(f'\n{dataset}: {len(all_acc)} total points (per-detector Pareto fronts)')

    # ── Per-detector distances ───────────────────────────────────────────
    ds_results = {}
    for det_name in sorted(detectors):
        mask = all_labels == det_name
        det_points = points_norm[mask]

        if USE_OWN_PARETO_FRONT:
            det_acc = all_acc[mask]
            det_rt = all_rt[mask]
            det_pareto_mask = compute_pareto_front(det_acc, det_rt)
            pareto_ref = det_points[det_pareto_mask]
            n_pareto = det_pareto_mask.sum()
        else:
            pareto_ref = global_pareto_points
            n_pareto = len(pareto_ref)

        dists = min_distance_to_pareto(det_points, pareto_ref)

        eps_fracs = {}
        for eps in EPSILONS:
            eps_fracs[eps] = float((dists <= eps).mean())

        ds_results[det_name] = {
            'distances': dists,
            'n_points': len(dists),
            'n_pareto': n_pareto,
            'mean_dist': float(dists.mean()),
            'median_dist': float(np.median(dists)),
            'p90_dist': float(np.percentile(dists, 90)),
            'eps_fractions': eps_fracs,
        }
        front_label = "own" if USE_OWN_PARETO_FRONT else "global"
        print(f'  {det_name:25s}  n={len(dists):5d}  pareto({front_label})={n_pareto:3d}  '
              f'mean={dists.mean():.4f}  median={np.median(dists):.4f}  '
              f'p90={np.percentile(dists, 90):.4f}  '
              f'ε=0.05 frac={eps_fracs[0.05]:.2%}')

    results_per_ds[dataset] = ds_results

## 4  Per-dataset summary tables

In [25]:
for dataset in sorted(results_per_ds):
    ds_results = results_per_ds[dataset]
    rows = []
    for det_name in sorted(ds_results):
        if det_name == "MoPEDDs_PreSelected":
            continue
        r = ds_results[det_name]
        row = {
            'Detector': det_name,
            'N': r['n_points'],
            'Mean Dist': r['mean_dist'],
            'Median Dist': r['median_dist'],
            '90th %ile Dist': r['p90_dist'],
        }
        for eps in EPSILONS:
            row[f'ρ(ε={eps})'] = r['eps_fractions'][eps]
        rows.append(row)

    tbl = pd.DataFrame(rows).set_index('Detector')
    print(f'\n{"="*80}')
    print(f'  Dataset: {dataset}')
    print(f'{"="*80}')
    display(tbl.style.format({
        'Mean Dist': '{:.4f}',
        'Median Dist': '{:.4f}',
        '90th %ile Dist': '{:.4f}',
        **{f'ρ(ε={eps})': '{:.2%}' for eps in EPSILONS},
    }).background_gradient(subset=['Mean Dist', 'Median Dist', '90th %ile Dist'],
                           cmap='RdYlGn_r')
     .background_gradient(subset=[f'ρ(ε={eps})' for eps in EPSILONS],
                          cmap='RdYlGn'))


  Dataset: Electricity


,N,Mean Dist,Median Dist,90th %ile Dist,ρ(ε=0.01),ρ(ε=0.025),ρ(ε=0.05),ρ(ε=0.1)
Detector,,,,,,,,
BNDM,4187,0.0373,0.0206,0.0775,22.62%,55.05%,74.78%,92.43%
CSDDM,4628,0.0379,0.0211,0.1014,6.05%,61.32%,77.16%,89.46%
D3,4633,0.0737,0.0490,0.1736,13.86%,34.04%,50.81%,70.30%
IBDD,3931,0.0943,0.0651,0.2103,3.18%,22.79%,41.31%,65.07%
MoPEDDs,4292,0.0282,0.0158,0.0676,36.79%,69.59%,83.43%,95.46%
OCDD,2111,0.0555,0.0306,0.1270,5.64%,44.15%,62.34%,80.20%
SPLL,5000,0.0636,0.0334,0.1630,16.52%,45.82%,57.08%,73.08%
UDetect,5000,0.0669,0.0251,0.2293,32.10%,49.88%,67.62%,77.12%



  Dataset: GasSensor


,N,Mean Dist,Median Dist,90th %ile Dist,ρ(ε=0.01),ρ(ε=0.025),ρ(ε=0.05),ρ(ε=0.1)
Detector,,,,,,,,
BNDM,770,0.0451,0.0219,0.1054,23.25%,55.32%,76.36%,89.48%
CSDDM,3922,0.0179,0.0137,0.0403,36.72%,73.56%,99.03%,99.85%
D3,4999,0.0136,0.0103,0.0318,48.55%,85.48%,99.48%,99.92%
IBDD,1617,0.0192,0.0157,0.0418,25.11%,73.84%,98.14%,100.00%
MoPEDDs,4997,0.0138,0.0067,0.0308,61.50%,86.77%,98.34%,98.90%
OCDD,2849,0.0188,0.0146,0.0405,34.05%,70.09%,99.75%,99.89%
SPLL,5000,0.0115,0.0096,0.0218,52.50%,91.74%,100.00%,100.00%
UDetect,2082,0.0200,0.0102,0.0416,49.81%,71.76%,95.58%,97.74%



  Dataset: PokerHand


,N,Mean Dist,Median Dist,90th %ile Dist,ρ(ε=0.01),ρ(ε=0.025),ρ(ε=0.05),ρ(ε=0.1)
Detector,,,,,,,,
BNDM,887,0.0270,0.0063,0.0381,75.65%,88.28%,90.30%,92.78%
CSDDM,892,0.0273,0.0174,0.0348,0.11%,70.29%,94.28%,97.76%
D3,2124,0.0133,0.0070,0.0228,66.53%,91.53%,96.56%,98.49%
IBDD,155,0.1883,0.1368,0.4166,0.00%,5.16%,17.42%,36.77%
MoPEDDs,537,0.1558,0.0186,0.5259,16.39%,55.31%,64.80%,69.65%
OCDD,624,0.0432,0.0182,0.0953,0.64%,66.83%,85.10%,90.54%
SPLL,1318,0.0202,0.0077,0.0347,67.15%,88.01%,91.88%,95.52%
UDetect,976,0.1074,0.0415,0.4206,36.37%,46.62%,51.23%,58.30%



  Dataset: RialtoBridgeTimelapse


,N,Mean Dist,Median Dist,90th %ile Dist,ρ(ε=0.01),ρ(ε=0.025),ρ(ε=0.05),ρ(ε=0.1)
Detector,,,,,,,,
BNDM,740,0.0344,0.0121,0.0681,41.08%,75.95%,88.92%,91.08%
CSDDM,3134,0.0140,0.0112,0.0301,45.47%,85.71%,99.59%,99.94%
D3,5000,0.0122,0.0097,0.0271,51.24%,87.54%,99.46%,100.00%
IBDD,2070,0.0221,0.0210,0.0415,23.48%,61.64%,99.57%,100.00%
MoPEDDs,200,0.0811,0.0239,0.2371,21.00%,51.00%,61.00%,72.00%
OCDD,1290,0.0220,0.0175,0.0443,25.81%,63.80%,95.89%,100.00%
SPLL,2685,0.0140,0.0125,0.0296,40.56%,84.10%,100.00%,100.00%
UDetect,2122,0.0232,0.0180,0.0573,33.36%,63.05%,85.82%,100.00%


## 5  Aggregate across datasets

In [26]:
# Build a DataFrame: one row per (detector, dataset) with the summary statistics
agg_rows = []
for dataset in sorted(results_per_ds):
    for det_name, r in results_per_ds[dataset].items():
        row = {
            'Dataset': dataset,
            'Detector': det_name,
            'Mean Dist': r['mean_dist'],
            'Median Dist': r['median_dist'],
            '90th %ile Dist': r['p90_dist'],
        }
        for eps in EPSILONS:
            row[f'ρ(ε={eps})'] = r['eps_fractions'][eps]
        agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)

# Cross-dataset mean per detector
cross_ds = agg_df.drop(columns='Dataset').groupby('Detector').mean()
print('Cross-dataset average (over datasets):')
display(cross_ds.style.format({
    'Mean Dist': '{:.4f}',
    'Median Dist': '{:.4f}',
    '90th %ile Dist': '{:.4f}',
    **{f'ρ(ε={eps})': '{:.2%}' for eps in EPSILONS},
}).background_gradient(subset=['Mean Dist', 'Median Dist', '90th %ile Dist'],
                       cmap='RdYlGn_r')
 .background_gradient(subset=[f'ρ(ε={eps})' for eps in EPSILONS],
                      cmap='RdYlGn'))

Cross-dataset average (over datasets):


,Mean Dist,Median Dist,90th %ile Dist,ρ(ε=0.01),ρ(ε=0.025),ρ(ε=0.05),ρ(ε=0.1)
Detector,,,,,,,
BNDM,0.0359,0.0152,0.0723,40.65%,68.65%,82.59%,91.44%
CSDDM,0.0243,0.0159,0.0516,22.09%,72.72%,92.51%,96.75%
D3,0.0282,0.0190,0.0638,45.04%,74.65%,86.58%,92.18%
IBDD,0.0810,0.0596,0.1775,12.94%,40.86%,64.11%,75.46%
MoPEDDs,0.0697,0.0162,0.2153,33.92%,65.67%,76.89%,84.00%
OCDD,0.0349,0.0202,0.0768,16.53%,61.22%,85.77%,92.66%
SPLL,0.0273,0.0158,0.0623,44.18%,77.42%,87.24%,92.15%
UDetect,0.0544,0.0237,0.1872,37.91%,57.83%,75.06%,83.29%


## 6  Box-whisker plots across datasets

Each box shows the distribution of a detector's per-dataset summary statistic.

In [27]:
# Collect *all* per-point distances, tagged by detector and dataset,
# so each box aggregates over all points across all datasets for that detector.
dist_rows = []
for dataset in sorted(results_per_ds):
    for det_name, r in results_per_ds[dataset].items():
        for d in r['distances']:
            dist_rows.append({'Detector': det_name, 'Dataset': dataset, 'Distance': d})

dist_df = pd.DataFrame(dist_rows)

# Determine detector order by median distance (most robust first)
det_order = (dist_df.groupby('Detector')['Distance']
             .median()
             .sort_values()
             .index
             .tolist())

print('Detector order (by median distance, ascending):', det_order)

Detector order (by median distance, ascending): ['MoPEDDs', 'D3', 'SPLL', 'BNDM', 'CSDDM', 'OCDD', 'UDetect', 'IBDD']


In [28]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left panel: Distance to Pareto Front ────────────────────────────────────
ax = axes[0]
box_data = [dist_df[dist_df['Detector'] == d]['Distance'].values for d in det_order]
bp = ax.boxplot(box_data, tick_labels=det_order, patch_artist=True,
                showfliers=False, widths=0.6,
                medianprops=dict(color='black', linewidth=1.5))

for patch, det_name in zip(bp['boxes'], det_order):
    color = DETECTOR_COLORS.get(det_name, '#aaaaaa')
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Normalised Euclidean distance', fontsize=12)
front_label = 'Own' if USE_OWN_PARETO_FRONT else 'Global'
ax.set_title(f'Per-Point Euclidean Distance to {front_label} Pareto Front\n(all configurations across all datasets)', fontsize=13)
ax.tick_params(axis='x', rotation=35)
ax.grid(True, axis='y', alpha=0.3)

# ── Right panel: ε-near-Pareto fraction per dataset ─────────────────────────
ax = axes[1]
eps_choice = 0.01  # default ε for the box plot
frac_data = []
for det_name in det_order:
    fracs = []
    for dataset in sorted(results_per_ds):
        if det_name in results_per_ds[dataset]:
            fracs.append(results_per_ds[dataset][det_name]['eps_fractions'][eps_choice])
    frac_data.append(fracs)

bp2 = ax.boxplot(frac_data, tick_labels=det_order, patch_artist=True,
                 showfliers=True, widths=0.6,
                 medianprops=dict(color='black', linewidth=1.5))

for patch, det_name in zip(bp2['boxes'], det_order):
    color = DETECTOR_COLORS.get(det_name, '#aaaaaa')
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel(f'Fraction within ε = {eps_choice}', fontsize=12)
ax.set_title(f'ε-Near-Pareto Fraction (ε = {eps_choice})\n(one value per detector per dataset)', fontsize=13)
ax.tick_params(axis='x', rotation=35)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(-0.05, 1.05)

fig.tight_layout()
fig.savefig('robustness_boxplots.pdf', bbox_inches='tight')
fig.savefig('robustness_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved robustness_boxplots.pdf and robustness_boxplots.png')

## 7  Additional box-whisker: per-dataset breakdown (Mean Distance)

In [29]:
# One box per detector, where each data point is that detector's mean distance
# on one dataset.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: mean distance per dataset ──────────────────────────────────────────
ax = axes[0]
mean_data = []
for det_name in det_order:
    vals = []
    for dataset in sorted(results_per_ds):
        if det_name in results_per_ds[dataset]:
            vals.append(results_per_ds[dataset][det_name]['mean_dist'])
    mean_data.append(vals)

bp3 = ax.boxplot(mean_data, tick_labels=det_order, patch_artist=True,
                 showfliers=True, widths=0.6,
                 medianprops=dict(color='black', linewidth=1.5))
for patch, det_name in zip(bp3['boxes'], det_order):
    color = DETECTOR_COLORS.get(det_name, '#aaaaaa')
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Mean normalised Euclidean distance', fontsize=12)
front_label = 'Own' if USE_OWN_PARETO_FRONT else 'Global'
ax.set_title(f'Mean Euclidean Distance to {front_label} Pareto Front\n(one value per detector per dataset)', fontsize=13)
ax.tick_params(axis='x', rotation=35)
ax.grid(True, axis='y', alpha=0.3)

# ── Right: 90th percentile distance per dataset ──────────────────────────────
ax = axes[1]
p90_data = []
for det_name in det_order:
    vals = []
    for dataset in sorted(results_per_ds):
        if det_name in results_per_ds[dataset]:
            vals.append(results_per_ds[dataset][det_name]['p90_dist'])
    p90_data.append(vals)

bp4 = ax.boxplot(p90_data, tick_labels=det_order, patch_artist=True,
                 showfliers=True, widths=0.6,
                 medianprops=dict(color='black', linewidth=1.5))
for patch, det_name in zip(bp4['boxes'], det_order):
    color = DETECTOR_COLORS.get(det_name, '#aaaaaa')
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('90th %ile normalised Euclidean distance', fontsize=12)
ax.set_title(f'90th Percentile Euclidean Distance to {front_label} Pareto Front\n(one value per detector per dataset)', fontsize=13)
ax.tick_params(axis='x', rotation=35)
ax.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig('robustness_boxplots_per_dataset.pdf', bbox_inches='tight')
fig.savefig('robustness_boxplots_per_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved robustness_boxplots_per_dataset.pdf and robustness_boxplots_per_dataset.png')

## 8  Export results to CSV

In [30]:
agg_df.to_csv('robustness_results.csv', index=False)
cross_ds.to_csv('robustness_cross_dataset.csv')
print('Saved robustness_results.csv and robustness_cross_dataset.csv')

Saved robustness_results.csv and robustness_cross_dataset.csv
